# Phase 1 Task B — Results Driver

Sau khi cả 5 notebooks đã chạy xong, chạy notebook này
để tổng hợp kết quả từ wandb.

Chạy được trên local / Colab / bất kỳ máy nào có internet.

In [1]:
from pathlib import Path
NOTEBOOK = Path("__file__").resolve()
ROOT = NOTEBOOK
for _ in range(10):
    ROOT = ROOT.parent
    if (ROOT / "pyproject.toml").exists() or (ROOT / ".git").exists(): break
import sys, os
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
print(f"[ROOT] {ROOT}")

[ROOT] /workspace/woodbench


In [2]:
from dotenv import load_dotenv
load_dotenv(dotenv_path := ROOT / ".env")
print(f"[.env] Loaded from {dotenv_path}")

import subprocess, sys
# for pkg in ["wandb", "scikit-learn", "seaborn", "matplotlib", "timm", "albumentations"]:
#     try: __import__(pkg.replace("-", "_"))
#     except: subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

os.environ.setdefault("WANDB_API_KEY", os.getenv("WANDB_API_KEY", ""))
print(f"[wandb] API key: {'set' if os.getenv('WANDB_API_KEY') else 'MISSING'}")

python-dotenv could not parse statement starting at line 4
python-dotenv could not parse statement starting at line 6
python-dotenv could not parse statement starting at line 7


[.env] Loaded from /workspace/woodbench/.env
[wandb] API key: set


In [3]:
import wandb
api = wandb.Api()
print("[wandb] API ready")

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


[wandb] API ready


## Fetch Runs from wandb

In [7]:
PROJECT = "woodbench-taskB"          # nếu cần: "your-entity/woodbench-taskB"
MODELS = ["convnext_tiny", "mobilenetv3_large", "swin_tiny", "resnet50", "efficientnet_b3"]
SEEDS  = [42, 123, 456]

runs_data = {}

for model in MODELS:
    runs_data[model] = {}
    for seed in SEEDS:
        try:
            runs = api.runs(
                PROJECT,
                filters={
                    "$and": [
                        {"config.backbone": model},
                        {"config.seed":     seed},
                    ]
                },
            )
            run_list = list(runs)          # ép về list để kiểm tra độ dài
            if run_list:
                run     = run_list[0]
                summary = dict(run.summary)          # ← sửa: thay _json bằng dict()
                runs_data[model][seed] = {
                    "name":         run.name,
                    "best_val_acc": summary.get("val/acc", 0),
                    "run_id":       run.id,
                    "state":        run.state,
                }
                print(f"[OK]      {model} s{seed}: {runs_data[model][seed]['best_val_acc']:.2f}%")
            else:
                print(f"[MISSING] {model} s{seed}")
        except Exception as e:
            print(f"[ERR]     {model} s{seed}: {e}")

[OK]      convnext_tiny s42: 0.00%
[OK]      convnext_tiny s123: 0.00%
[OK]      convnext_tiny s456: 0.00%
[OK]      mobilenetv3_large s42: 0.00%
[OK]      mobilenetv3_large s123: 0.00%
[OK]      mobilenetv3_large s456: 0.00%
[OK]      swin_tiny s42: 0.00%
[OK]      swin_tiny s123: 0.00%
[OK]      swin_tiny s456: 0.00%
[OK]      resnet50 s42: 0.00%
[OK]      resnet50 s123: 0.00%
[OK]      resnet50 s456: 0.00%
[OK]      efficientnet_b3 s42: 0.00%
[OK]      efficientnet_b3 s123: 0.00%
[OK]      efficientnet_b3 s456: 0.00%


## Summary Table

In [9]:
# Debug: xem summary của run đầu tiên tìm được
for model in MODELS:
    run_list = list(api.runs(PROJECT, filters={"config.backbone": model}))
    if run_list:
        s = dict(run_list[0].summary)
        print(f"\n[{model}] summary keys:")
        for k, v in s.items():
            if not k.startswith("_"):   # bỏ qua meta keys
                print(f"  {k}: {v}")
        break


[convnext_tiny] summary keys:


In [10]:
for model in MODELS:
    run_list = list(api.runs(PROJECT, filters={"config.backbone": model}))
    if run_list:
        run = run_list[0]
        print(f"\n[{model}] state={run.state}  name={run.name}")
        
        # Xem history có metric không
        hist = run.history(samples=5)
        print(f"  history columns: {list(hist.columns)}")
        print(hist.tail(3).to_string())


[convnext_tiny] state=finished  name=convnext_tiny_s42
  history columns: []
Empty DataFrame
Columns: []
Index: []

[mobilenetv3_large] state=crashed  name=mobilenetv3_large_s42
  history columns: []
Empty DataFrame
Columns: []
Index: []

[swin_tiny] state=finished  name=swin_tiny_s42
  history columns: []
Empty DataFrame
Columns: []
Index: []

[resnet50] state=finished  name=resnet50_s42
  history columns: []
Empty DataFrame
Columns: []
Index: []

[efficientnet_b3] state=crashed  name=efficientnet_b3_s42
  history columns: []
Empty DataFrame
Columns: []
Index: []


In [11]:
for model in MODELS:
    run_list = list(api.runs(PROJECT, filters={"config.backbone": model}))
    if run_list:
        run = run_list[0]
        print(f"\n[{model}] state={run.state}")
        print(f"  config: {dict(run.config)}")


[convnext_tiny] state=finished
  config: {'amp': True, 'seed': 42, 'epochs': 50, 'lr_head': 0.0001, 'backbone': 'convnext_tiny', 'scheduler': 'cosine', 'batch_size': 32, 'lr_backbone': 1e-05, 'num_classes': 20, 'label_smoothing': 0.1}

[mobilenetv3_large] state=crashed
  config: {'amp': True, 'seed': 42, 'epochs': 50, 'lr_head': 0.0001, 'backbone': 'mobilenetv3_large', 'scheduler': 'cosine', 'batch_size': 64, 'lr_backbone': 1e-05, 'num_classes': 20, 'label_smoothing': 0.1}

[swin_tiny] state=finished
  config: {'amp': True, 'seed': 42, 'epochs': 50, 'lr_head': 0.0001, 'backbone': 'swin_tiny', 'scheduler': 'cosine', 'batch_size': 32, 'lr_backbone': 1e-05, 'num_classes': 20, 'label_smoothing': 0.1}

[resnet50] state=finished
  config: {'amp': True, 'seed': 42, 'epochs': 50, 'lr_head': 0.0001, 'backbone': 'resnet50', 'scheduler': 'cosine', 'batch_size': 64, 'lr_backbone': 1e-05, 'num_classes': 20, 'label_smoothing': 0.1}

[efficientnet_b3] state=crashed
  config: {'amp': True, 'seed': 42

In [12]:
import torch
from pathlib import Path

ROOT_EXP = ROOT / "data" / "experiments" / "taskB"
MODELS = ["convnext_tiny", "mobilenetv3_large", "swin_tiny", "resnet50", "efficientnet_b3"]
SEEDS  = [42, 123, 456]

# Xem keys trong 1 checkpoint đầu tiên tìm được
for model in MODELS:
    for seed in SEEDS:
        ckpt_dir = ROOT_EXP / model / f"seed_{seed}"
        candidates = sorted(ckpt_dir.glob("*best*.pth"))
        if candidates:
            ckpt = torch.load(candidates[0], map_location="cpu")
            print(f"[{model} s{seed}] keys: {list(ckpt.keys())}")
            break
    else:
        continue
    break

[convnext_tiny s42] keys: ['epoch', 'model_state_dict', 'optimizer_state_dict', 'metrics', 'best_val_acc', 'best_epoch', 'scheduler_state_dict']


In [14]:
import torch
import pandas as pd
import numpy as np
from pathlib import Path

ROOT_EXP = ROOT / "data" / "experiments" / "taskB"
MODELS = ["convnext_tiny", "mobilenetv3_large", "swin_tiny", "resnet50", "efficientnet_b3"]
SEEDS  = [42, 123, 456]

# ── Đọc kết quả từ checkpoint ──────────────────────────────────────────────
rows = []
for model in MODELS:
    for seed in SEEDS:
        ckpt_dir = ROOT_EXP / model / f"seed_{seed}"
        candidates = sorted(ckpt_dir.glob("*best*.pth"))
        if candidates:
            ckpt = torch.load(candidates[0], map_location="cpu")
            val_acc = ckpt["best_val_acc"]
            epoch   = ckpt.get("best_epoch", "?")
            rows.append({
                "Model":            model,
                "Seed":             seed,
                "Best Val Acc (%)": round(float(val_acc), 2),
                "Best Epoch":       epoch,
                "Status":           "ok",
            })
            print(f"[OK] {model} s{seed}: {val_acc:.2f}%  (epoch {epoch})")
        else:
            rows.append({"Model": model, "Seed": seed,
                         "Best Val Acc (%)": 0.0, "Best Epoch": -1, "Status": "missing"})
            print(f"[MISSING] {model} s{seed}: no checkpoint found in {ckpt_dir}")

# ── Summary table ──────────────────────────────────────────────────────────
df = pd.DataFrame(rows)
summary = df.groupby("Model").agg(
    Mean_Acc =("Best Val Acc (%)", "mean"),
    Std_Acc  =("Best Val Acc (%)", "std"),
    Min_Acc  =("Best Val Acc (%)", "min"),
    Max_Acc  =("Best Val Acc (%)", "max"),
).round(2)

print("\n" + "="*70)
print("TASK B — 5 MODELS × 3 SEEDS SUMMARY")
print("="*70)
print(df.to_string(index=False))
print("\n" + "-"*70)
print("PER-MODEL SUMMARY")
print("-"*70)
print(summary.to_string())

best_model = summary["Mean_Acc"].idxmax()
best_acc   = summary.loc[best_model, "Mean_Acc"]
print(f"\nBest model: {best_model} ({best_acc:.2f}%)")

# ── Lưu CSV ───────────────────────────────────────────────────────────────
out_dir = ROOT /"data"/ "experiments" / "taskB"
out_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(out_dir / "results_summary.csv", index=False)
summary.to_csv(out_dir / "results_per_model.csv")
print(f"\n[Saved] {out_dir / 'results_summary.csv'}")
print("\n" + df.to_markdown(index=False))

[OK] convnext_tiny s42: 100.00%  (epoch 12)
[OK] convnext_tiny s123: 100.00%  (epoch 14)
[OK] convnext_tiny s456: 100.00%  (epoch 9)
[OK] mobilenetv3_large s42: 99.57%  (epoch 34)
[OK] mobilenetv3_large s123: 99.79%  (epoch 34)
[OK] mobilenetv3_large s456: 99.68%  (epoch 45)
[OK] swin_tiny s42: 100.00%  (epoch 6)
[OK] swin_tiny s123: 100.00%  (epoch 11)
[OK] swin_tiny s456: 100.00%  (epoch 9)
[OK] resnet50 s42: 95.39%  (epoch 39)
[OK] resnet50 s123: 96.03%  (epoch 49)
[OK] resnet50 s456: 96.14%  (epoch 44)
[OK] efficientnet_b3 s42: 99.36%  (epoch 39)
[OK] efficientnet_b3 s123: 99.46%  (epoch 35)
[OK] efficientnet_b3 s456: 99.46%  (epoch 36)

TASK B — 5 MODELS × 3 SEEDS SUMMARY
            Model  Seed  Best Val Acc (%)  Best Epoch Status
    convnext_tiny    42            100.00          12     ok
    convnext_tiny   123            100.00          14     ok
    convnext_tiny   456            100.00           9     ok
mobilenetv3_large    42             99.57          34     ok
mobilenet

In [8]:
import pandas as pd
import numpy as np

rows = []
for model in MODELS:
    for seed in SEEDS:
        d = runs_data[model].get(seed, {})
        rows.append({
            "Model": model,
            "Seed": seed,
            "Best Val Acc (%)": d.get("best_val_acc", 0),
            "State": d.get("state", "missing"),
        })

df = pd.DataFrame(rows)

summary = df.groupby("Model").agg(
    Mean_Acc=("Best Val Acc (%)", "mean"),
    Std_Acc=("Best Val Acc (%)", "std"),
    Min_Acc=("Best Val Acc (%)", "min"),
    Max_Acc=("Best Val Acc (%)", "max"),
).round(2)

print("\n" + "="*70)
print("TASK B — 5 MODELS × 3 SEEDS SUMMARY")
print("="*70)
print(df.to_string(index=False))
print("\n" + "-"*70)
print("PER-MODEL SUMMARY")
print("-"*70)
print(summary.to_string())

best_model = summary["Mean_Acc"].idxmax()
best_acc = summary.loc[best_model, "Mean_Acc"]
print(f"\nBest model: {best_model} ({best_acc:.2f}%)")


TASK B — 5 MODELS × 3 SEEDS SUMMARY
            Model  Seed  Best Val Acc (%)    State
    convnext_tiny    42                 0 finished
    convnext_tiny   123                 0 finished
    convnext_tiny   456                 0 finished
mobilenetv3_large    42                 0  crashed
mobilenetv3_large   123                 0 finished
mobilenetv3_large   456                 0 finished
        swin_tiny    42                 0 finished
        swin_tiny   123                 0 finished
        swin_tiny   456                 0 finished
         resnet50    42                 0 finished
         resnet50   123                 0 finished
         resnet50   456                 0 finished
  efficientnet_b3    42                 0  crashed
  efficientnet_b3   123                 0 finished
  efficientnet_b3   456                 0 finished

----------------------------------------------------------------------
PER-MODEL SUMMARY
---------------------------------------------------------

In [6]:
out_dir = ROOT / "experiments" / "taskB"
out_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(out_dir / "results_summary.csv", index=False)
summary.to_csv(out_dir / "results_per_model.csv")
print(f"[Saved] {out_dir / 'results_summary.csv'}")
print("\n" + df.to_markdown())
print("\n[Next] Run evaluate.py for full test-set metrics + confusion matrix")

[Saved] /workspace/woodbench/experiments/taskB/results_summary.csv

|    | Model             |   Seed |   Best Val Acc (%) | State   |
|---:|:------------------|-------:|-------------------:|:--------|
|  0 | convnext_tiny     |     42 |                  0 | missing |
|  1 | convnext_tiny     |    123 |                  0 | missing |
|  2 | convnext_tiny     |    456 |                  0 | missing |
|  3 | mobilenetv3_large |     42 |                  0 | missing |
|  4 | mobilenetv3_large |    123 |                  0 | missing |
|  5 | mobilenetv3_large |    456 |                  0 | missing |
|  6 | swin_tiny         |     42 |                  0 | missing |
|  7 | swin_tiny         |    123 |                  0 | missing |
|  8 | swin_tiny         |    456 |                  0 | missing |
|  9 | resnet50          |     42 |                  0 | missing |
| 10 | resnet50          |    123 |                  0 | missing |
| 11 | resnet50          |    456 |                  0 | miss